# 🧠 YOLO Grid System and Multi-Scale Cell Assignment

Welcome to the hands-on explanation notebook for the **YOLO Grid System**! In this notebook, we will:
1. Learn how YOLO divides input images into grids of different resolutions to detect objects.
2. Implement a cell assignment algorithm to determine which scale and grid cell is responsible for detecting specific objects.
3. Simulate three detection scales: Stride 8 ($80\times80$), Stride 16 ($40\times40$), and Stride 32 ($20\times20$).
4. Assign custom objects (like valves and wellheads) to their correct cells based on center points and areas.
5. Plot the grids and object mappings using Matplotlib to visually illustrate the multi-scale target assignment concept.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

## 1. Grid Cell Assignment Logic

This function takes bounding box coordinates and assigns the object to a specific scale grid cell.

In [ ]:
def assign_grid_cell(bbox, image_size=(640, 640)):
    x1, y1, x2, y2 = bbox
    w = x2 - x1
    h = y2 - y1
    area = w * h
    
    cx = (x1 + x2) / 2.0
    cy = (y1 + y2) / 2.0
    
    # Stride selection based on area
    if area < 64 ** 2:
        stride = 8
    elif area < 192 ** 2:
        stride = 16
    else:
        stride = 32
        
    cell_col = int(cx // stride)
    cell_row = int(cy // stride)
    grid_w = image_size[0] // stride
    grid_h = image_size[1] // stride
    
    return {
        "stride": stride,
        "grid_shape": (grid_w, grid_h),
        "cell_index": (cell_col, cell_row),
        "center_pixel": (cx, cy)
    }

## 2. Setting Up Test Objects

We set up three components from the PTT dataset with different dimensions and map them to their corresponding grids.

In [ ]:
test_objects = {
    "small-valve": [100.0, 100.0, 140.0, 130.0],  # Small
    "control-valve": [200.0, 200.0, 310.0, 320.0], # Medium
    "wellhead": [150.0, 150.0, 480.0, 520.0]       # Large
}

for name, bbox in test_objects.items():
    res = assign_grid_cell(bbox)
    print(f"{name.upper()}:\n  Assigned to stride {res['stride']} grid {res['grid_shape']} cell {res['cell_index']} at center {res['center_pixel']}\n")

## 3. Visualizing Grid Mappings

We plot each object against its assigned grid layer. The red dot represents the object center, and the highlighted blue box represents the responsible grid cell.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
names = ["small-valve", "control-valve", "wellhead"]
colors = ["green", "blue", "purple"]

for idx, ax in enumerate(axes):
    name = names[idx]
    bbox = test_objects[name]
    res = assign_grid_cell(bbox)
    stride = res["stride"]
    cell_col, cell_row = res["cell_index"]
    cx, cy = res["center_pixel"]
    
    ax.set_xlim(0, 640)
    ax.set_ylim(640, 0)
    
    # Draw object bbox
    w = bbox[2] - bbox[0]
    h = bbox[3] - bbox[1]
    rect = patches.Rectangle((bbox[0], bbox[1]), w, h, linewidth=2, edgecolor=colors[idx], facecolor='none', label=name)
    ax.add_patch(rect)
    
    # Draw responsible cell grid boundary
    cell_x1 = cell_col * stride
    cell_y1 = cell_row * stride
    rect_cell = patches.Rectangle((cell_x1, cell_y1), stride, stride, linewidth=2, edgecolor='red', facecolor='red', alpha=0.3, label='Responsible Grid Cell')
    ax.add_patch(rect_cell)
    
    # Plot center point
    ax.plot(cx, cy, 'ro', markersize=8, label='Object Center')
    
    # Add subset of grid lines for visualization
    for val in range(0, 640, stride * 2):
        ax.axhline(val, color='gray', linestyle=':', alpha=0.3)
        ax.axvline(val, color='gray', linestyle=':', alpha=0.3)
        
    ax.set_title(f"{name} on Stride {stride} Grid", fontsize=12)
    ax.legend()
    ax.grid(False)

plt.tight_layout()
plt.show()

## 4. Key Takeaways

-   **Single Cell Assignment:** Even if an object is large and covers many cells, only **one** cell (the one containing the center point) is responsible for predicting it. This avoids duplicate training targets.
-   **Scale Aggregation:** Having multiple grids allows YOLO to optimize its feature maps: large grids (stride 8) retain localized details, whereas small grids (stride 32) retain global context.